In [1]:
import os
import random
import time

import pandas as pd
from dotenv import load_dotenv
from nba_api.stats.endpoints import leaguegamefinder
from sqlalchemy import create_engine, inspect


# Helper functions
def normalize_for_postgres(df):
    """Normalize DataFrame column names for PostgreSQL"""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df


load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# --- NEW: Check for existing games to avoid duplicates ---
inspector = inspect(engine)
existing_game_ids = set()

if "team_game_stats" in inspector.get_table_names():
    print("Fetching existing game IDs from database...")
    existing_games = pd.read_sql("SELECT DISTINCT game_id FROM team_game_stats", engine)
    existing_game_ids = set(existing_games["game_id"].unique())
    print(f"Found {len(existing_game_ids)} games already in database.")

seasons = ["2024-25"]
all_seasons_data = []

for season in seasons:
    try:
        print(f"Fetching games for {season}...")
        game_finder = leaguegamefinder.LeagueGameFinder(
            season_nullable=season, league_id_nullable="00", season_type_nullable="Regular Season"
        )

        games_df = game_finder.get_data_frames()[0]
        games_df["SEASON"] = season

        # --- NEW: Filter out games we already have ---
        # Note: NBA API game_id is usually a string, ensure types match
        new_games_df = games_df[~games_df["GAME_ID"].isin(existing_game_ids)]

        if not new_games_df.empty:
            all_seasons_data.append(new_games_df)
            print(f"✓ Found {len(new_games_df)} new game records for {season}")
        else:
            print(f"- No new games found for {season}")

        time.sleep(round(random.uniform(1, 3), 1))

    except Exception as e:
        print(f"✗ Error for {season}: {e}")

# Process and Push
if all_seasons_data:
    # 1. Combine all seasons into one giant dataframe
    print("\nCombining collected seasons into single dataframe...")
    combined_df = pd.concat(all_seasons_data, ignore_index=True)

    # 2. Normalize and Map to Database Schema
    print("Aligning data with existing database schema...")
    combined_df = normalize_for_postgres(combined_df)

    # This mapping connects NBA API names to your 'team_' prefixed database columns
    mapping = {
        "game_date": "team_game_date",
        "matchup": "team_matchup",
        "wl": "team_wl",
        "min": "team_min",
        "pts": "team_pts",
        "fgm": "team_fgm",
        "fga": "team_fga",
        "fg_pct": "team_fg_pct",
        "fg3m": "team_fg3m",
        "fg3a": "team_fg3a",
        "fg3_pct": "team_fg3_pct",
        "ftm": "team_ftm",
        "fta": "team_fta",
        "ft_pct": "team_ft_pct",
        "oreb": "team_oreb",
        "dreb": "team_dreb",
        "reb": "team_reb",
        "ast": "team_ast",
        "stl": "team_stl",
        "blk": "team_blk",
        "tov": "team_tov",
        "pf": "team_pf",
        "plus_minus": "team_plus_minus",
    }
    combined_df = combined_df.rename(columns=mapping)

    # 3. Filter for columns the DB actually has
    # This ensures we don't try to insert extra API columns that weren't in your original table
    db_cols = [
        "season_id",
        "team_id",
        "team_abbreviation",
        "team_name",
        "game_id",
        "team_game_date",
        "team_matchup",
        "team_wl",
        "team_min",
        "team_pts",
        "team_fgm",
        "team_fga",
        "team_fg_pct",
        "team_fg3m",
        "team_fg3a",
        "team_fg3_pct",
        "team_ftm",
        "team_fta",
        "team_ft_pct",
        "team_oreb",
        "team_dreb",
        "team_reb",
        "team_ast",
        "team_stl",
        "team_blk",
        "team_tov",
        "team_pf",
        "team_plus_minus",
        "season",
    ]

    final_df = combined_df[combined_df.columns.intersection(db_cols)]

    # 4. Push to SQL using 'append'
    print(f"\nPushing {len(final_df)} records to 'team_game_stats'...")

    try:
        final_df.to_sql("team_game_stats", engine, if_exists="append", index=False)
        print("✅ Successfully updated team_game_stats!")
    except Exception as e:
        print(f"❌ Error during SQL push: {e}")
        # If it fails with a 'PendingRollbackError', run this:
        # engine.connect().rollback()

    print("\n✓ Finished! All data processed.")
else:
    print("\n🙌 No new team data to add!")

Fetching existing game IDs from database...
Found 19118 games already in database.
Fetching games for 2024-25...
- No new games found for 2024-25

🙌 No new team data to add!


In [7]:
# Run this to clear the 'failed' state
engine.connect().rollback()

In [6]:
from sqlalchemy import inspect

inspector = inspect(engine)
columns = [col["name"] for col in inspector.get_columns("team_game_stats")]
print("Actual DB Columns:", columns)

Actual DB Columns: ['season_id', 'team_id', 'team_abbreviation', 'team_name', 'game_id', 'team_game_date', 'team_matchup', 'team_wl', 'team_min', 'team_pts', 'team_fgm', 'team_fga', 'team_fg_pct', 'team_fg3m', 'team_fg3a', 'team_fg3_pct', 'team_ftm', 'team_fta', 'team_ft_pct', 'team_oreb', 'team_dreb', 'team_reb', 'team_ast', 'team_stl', 'team_blk', 'team_tov', 'team_pf', 'team_plus_minus', 'season']
